# Recalculation
Recalculate points from a list of known physical points

In [ ]:
# Imports
import os
import glob
from pathlib import Path
from typing import List
import pickle
import pandas as pd
import numpy as np
# == importar la libreria hecha ==
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", ".."))) # Añadir la ruta absoluta a 'mlpython/' al comienzo de sys.path
#import lib  
from lib.oracle import OracleExecutor


def discover_csv_files(pattern: str = "./known_valid_points/filtered_data_*.csv") -> List[Path]:
    """
    Discover CSV files matching the given glob pattern.
    Single Responsibility: I/O - file discovery
    """
    return [Path(fp) for fp in glob.glob(pattern)]


def load_base_points(csv_files: List[Path]) -> np.ndarray:
    """
    Load base parameter points from given CSV files.
    Each CSV should contain columns: m_phi, m_A, sin_ba, tan_beta, lambda_6, lambda_7, m12_2

    Returns:
        points: ndarray of shape (n_points, 7)
    """
    records = []
    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        cols = ["m_phi", "m_A", "sin_ba", "tan_beta", "lambda_6", "lambda_7", "m12_2"]
        # Validate presence of columns
        missing = set(cols) - set(df.columns)
        if missing:
            raise KeyError(f"Missing columns {missing} in {csv_path}")
        records.append(df[cols].values)
    if not records:
        return np.empty((0, 7))
    return np.vstack(records)


def run_oracle_for_points(
    points: np.ndarray,
    executor: OracleExecutor,
    outdir: Path = Path("../../data_baches"),
    batch_size: int = 1000
) -> None:
    """
    Execute OracleExecutor on provided points in batches.
    Splits the points into chunks of size batch_size, submits them,
    and saves results as pickled files in outdir.
    """
    outdir.mkdir(parents=True, exist_ok=True)
    total = len(points)
    for i in range(0, total, batch_size):
        batch = points[i : i + batch_size]
        batch_idx = len(list(outdir.glob("batch_*.pkl"))) + 1
        print(f"Running batch {batch_idx}: points {i}–{i + len(batch)}")
        results = executor.map(batch.tolist(), use_threads=True)
        fname = outdir / f"batch_{batch_idx}.pkl"
        with open(fname, "wb") as f:
            pickle.dump({"params": batch, "results": results}, f)
        print(f"Saved: {fname}\n")


def main(
    csv_pattern: str = "./output/filtered_data_*.csv",
    n_threads: int = 4,
    batch_size: int = 1000
) -> None:
    """
    Main routine: discovers CSVs, loads points, and recalculates oracle results.
    """
    csv_files = discover_csv_files(csv_pattern)
    base_points = load_base_points(csv_files)
    print(f"Discovered {len(csv_files)} files, loaded {len(base_points)} points.")

    executor = OracleExecutor(nthreads=n_threads)
    run_oracle_for_points(
        points=base_points,
        executor=executor,
        outdir=Path("recalc_batches"),
        batch_size=batch_size
    )



In [4]:
main("./known_valid_points/filtered_data_*.csv", 4, 5_000)

Discovered 6 files, loaded 115744 points.
Running batch 1: points 0–5000
Saved: recalc_batches/batch_1.pkl

Running batch 2: points 5000–10000
Saved: recalc_batches/batch_2.pkl

Running batch 3: points 10000–15000
Saved: recalc_batches/batch_3.pkl

Running batch 4: points 15000–20000
Saved: recalc_batches/batch_4.pkl

Running batch 5: points 20000–25000
Saved: recalc_batches/batch_5.pkl

Running batch 6: points 25000–30000
Saved: recalc_batches/batch_6.pkl

Running batch 7: points 30000–35000
Saved: recalc_batches/batch_7.pkl

Running batch 8: points 35000–40000
Saved: recalc_batches/batch_8.pkl

Running batch 9: points 40000–45000
Saved: recalc_batches/batch_9.pkl

Running batch 10: points 45000–50000
Saved: recalc_batches/batch_10.pkl

Running batch 11: points 50000–55000
Saved: recalc_batches/batch_11.pkl

Running batch 12: points 55000–60000
Saved: recalc_batches/batch_12.pkl

Running batch 13: points 60000–65000
Saved: recalc_batches/batch_13.pkl

Running batch 14: points 65000–70